# Sampling Controls

Control the creativity and randomness of LLM outputs.

**Temperature**, **Top-p**, **Top-k**, and **Seed** — the knobs that shape how the model generates text.

## Setup

In [ ]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GEMINI_API_KEY2"]
client = genai.Client(api_key=API_KEY)

## Temperature

Controls randomness. Higher = more creative/random. Lower = more focused/deterministic.

- **0.0** — Deterministic, always picks the most likely token
- **0.5** — Balanced
- **1.0** — Default, good balance
- **2.0** — Very creative, may be incoherent

In [ ]:
def generate(prompt, temperature=1.0):
    """Generate text with specified temperature."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text

prompt = "Write a one-sentence story about a robot."

print("=== Temperature 0.0 (Deterministic) ===")
for i in range(2):
    print(f"{i+1}. {generate(prompt, temperature=0.0)}")

In [ ]:
# Compare temperatures
prompt = "Write a one-sentence story about a robot."

for temp in [0.5, 1.0, 1.5]:
    print(f"\n=== Temperature {temp} ===")
    print(generate(prompt, temperature=temp))

## Top-p (Nucleus Sampling)

Keep adding tokens until their **cumulative probability** reaches p.

**Example:** Tokens ranked by probability:
```
Token A: 40% → cumulative: 40%
Token B: 30% → cumulative: 70%
Token C: 15% → cumulative: 85%
Token D: 10% → cumulative: 95%
Token E:  5% → cumulative: 100%
```

- **Top-p 0.5** → Only ~2 tokens (A, partial B) → **Focused**
- **Top-p 0.95** → 4 tokens (A, B, C, D) → **More diverse**

**Lower p = fewer tokens = more focused**

In [ ]:
def generate_topp(prompt, top_p=1.0):
    """Generate text with specified top_p."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_p=top_p,
            temperature=1.0  # Keep temperature constant
        )
    )
    return response.text

prompt = "List 5 creative uses for a paperclip:"

print("=== Top-p 0.5 (Focused) ===")
print(generate_topp(prompt, top_p=0.5))

print("\n=== Top-p 0.95 (More Creative) ===")
print(generate_topp(prompt, top_p=0.95))

## Top-k

Only consider the **top k most likely tokens** (by count, not probability).

**Example:** Tokens ranked by probability:
```
Token 1: 35%  ← Top 1
Token 2: 25%  ← Top 2
Token 3: 15%  ← Top 3
Token 4: 10%  ← Top 4
Token 5:  8%  ← Top 5
Token 6:  4%
Token 7:  2%
...
```

- **Top-k 1** → Only Token 1 → **Greedy (deterministic)**
- **Top-k 3** → Tokens 1, 2, 3 → **Focused**
- **Top-k 40** → Top 40 tokens → **Balanced (default)**
- **Top-k 100** → Top 100 tokens → **More diverse**

**Lower k = fewer tokens = more focused**

**Top-k vs Top-p:**
- Top-k: Fixed number of tokens
- Top-p: Variable number based on cumulative probability

In [ ]:
def generate_topk(prompt, top_k=40):
    """Generate text with specified top_k."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_k=top_k,
            temperature=1.0
        )
    )
    return response.text

prompt = "Invent a new word and define it:"

print("=== Top-k 1 (Greedy) ===")
print(generate_topk(prompt, top_k=1))

print("\n=== Top-k 40 (Default) ===")
print(generate_topk(prompt, top_k=40))

## Seed (Reproducibility)

Set a seed for reproducible outputs. Same seed + same prompt = same output.

Useful for:
- Testing and debugging
- Reproducible experiments
- Consistent outputs in production

In [ ]:
def generate_with_seed(prompt, seed=None):
    """Generate text with optional seed for reproducibility."""
    config = types.GenerateContentConfig(
        temperature=1.0
    )
    if seed is not None:
        config.seed = seed

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Tell me a joke about programming."

print("=== Without seed (different each time) ===")
print(f"1. {generate_with_seed(prompt)}")
print(f"2. {generate_with_seed(prompt)}")

print("\n=== With seed 42 (reproducible) ===")
print(f"1. {generate_with_seed(prompt, seed=42)}")
print(f"2. {generate_with_seed(prompt, seed=42)}")

## Combining Parameters

In [ ]:
def generate_custom(prompt, temperature=1.0, top_p=0.95, top_k=40, seed=None):
    """Generate with all sampling controls."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    if seed is not None:
        config.seed = seed

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Write a haiku about AI:"

# Focused, deterministic
print("=== Focused (temp=0.3, top_p=0.5, top_k=10) ===")
print(generate_custom(prompt, temperature=0.3, top_p=0.5, top_k=10))

# Creative, diverse
print("\n=== Creative (temp=1.5, top_p=0.95, top_k=100) ===")
print(generate_custom(prompt, temperature=1.5, top_p=0.95, top_k=100))

## Use Case Presets

In [ ]:
# Define presets for different use cases
PRESETS = {
    "factual": {"temperature": 0.0, "top_p": 0.5, "top_k": 10},
    "balanced": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "creative": {"temperature": 1.2, "top_p": 0.95, "top_k": 100},
    "brainstorm": {"temperature": 1.5, "top_p": 0.98, "top_k": 150},
}

prompt = "What are some ways to improve productivity?"

for name, params in PRESETS.items():
    print(f"\n=== {name.upper()} ===")
    print(generate_custom(prompt, **params))

## Key Takeaways

| Parameter | What it does | Low value | High value |
|-----------|-------------|-----------|------------|
| **Temperature** | Controls randomness | Focused, deterministic | Creative, random |
| **Top-p** | Nucleus sampling | Only top tokens | More diverse tokens |
| **Top-k** | Limits token choices | Few choices (greedy) | Many choices |
| **Seed** | Reproducibility | Random each time | Same output |

**Common combinations:**
- **Factual/Code**: temp=0, top_p=0.5, top_k=10
- **General chat**: temp=0.7, top_p=0.9, top_k=40
- **Creative writing**: temp=1.2, top_p=0.95, top_k=100
- **Brainstorming**: temp=1.5, top_p=0.98, top_k=150

---

**Tip:** Temperature is the most important. Start there, then fine-tune with top_p and top_k.